# AppAgentica: Consulente RAG per Algoritmi di Ordinamento

Questo notebook illustra l'architettura completa del sistema RAG:
1. **Catalogo algoritmi** (`data/algorithms.json`)
2. **Embedding locali su CPU** tramite FastEmbed (`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`)
3. **Vector Database Qdrant** (in esecuzione su Docker su `http://localhost:6333`)
4. **Prompt Engine & RAG** con LLM tramite OpenRouter (`openrouter/free`)

> **Prerequisiti**:
> - Qdrant avviato con `docker compose up -d`
> - File `.env` configurato con la chiave `OPENROUTER_API_KEY`

## 1. Setup dell'ambiente e importazioni

In [ ]:
import sys
from pathlib import Path

# Assicura la corretta risoluzione del package src/appagentica
ROOT_DIR = Path.cwd().resolve()
if (ROOT_DIR / "src").exists():
    sys.path.insert(0, str(ROOT_DIR / "src"))
elif (ROOT_DIR.parent / "src").exists():
    sys.path.insert(0, str(ROOT_DIR.parent / "src"))

from appagentica import config
from appagentica.ingest import load_dataset, build_text, ingest_dataset
from appagentica.embeddings import embed_query
from appagentica.rag import make_clients, retrieve, ask, format_context

print("AppAgentica importata con successo!")
print(f"Dataset: {config.DATA_FILE}")
print(f"Qdrant: {config.QDRANT_URL} (Collection: {config.COLLECTION_NAME})")
print(f"Modello Embedding: {config.EMBEDDING_MODEL}")
print(f"Modello Chat LLM: {config.CHAT_MODEL}")

## 2. Esplorazione del Dataset degli Algoritmi

In [ ]:
items = load_dataset()
print(f"Totale schede caricate: {len(items)}\n")

# Linguaggi coperti
languages = {}
for item in items:
    lang = item["language"]
    languages[lang] = languages.get(lang, 0) + 1

print("Distribuzione per linguaggio:")
for lang, count in sorted(languages.items(), key=lambda x: -x[1]):
    print(f"  - {lang}: {count} schede")

# Visualizza la prima scheda d'esempio
sample = items[0]
print(f"\nEsempio scheda #1: {sample['name']} [{sample['language']}]")
print(f"Algoritmo base: {sample['algorithm']}")
print(f"Complessità: media {sample['time_avg']}, peggiore {sample['time_worst']}, memoria {sample['space']}")
print(f"Quando usarlo: {sample['use_when']}")

## 3. Indicizzazione del Catalogo in Qdrant (Embedding Locali)

Gli embedding vengono generati **interamente in locale** su CPU tramite FastEmbed, senza costi o limiti di token.

In [ ]:
# Esegue l'indicizzazione completa su Qdrant
count = ingest_dataset()
print(f"Indicizzate {count} schede con successo nella collection '{config.COLLECTION_NAME}'!")

## 4. Ricerca Semantica Vettoriale (Retrieval)

Testiamo il recupero vettoriale senza ancora interrogare il modello linguistico, verificando i punteggi di similarità coseno.

In [ ]:
query_test = "ordinare 10 milioni di interi in Java sfruttando tutti i core della CPU"
points = retrieve(query_test, top_k=3)

print(f"Query: '{query_test}'\n")
print("Top 3 risultati per affinità vettoriale:")
for idx, p in enumerate(points, 1):
    payload = p.payload
    print(f"{idx}. Score: {p.score:.4f} | {payload['name']} [{payload['language']}]")
    print(f"   Algoritmo: {payload['algorithm']} | Complessità: {payload['time_avg']}")
    print(f"   Uso: {payload['use_when']}\n")

## 5. RAG End-to-End con OpenRouter LLM

Interroghiamo ora la pipeline completa con la funzione `ask()`: il sistema recupera le schede pertinenti e richiede al modello linguistico una spiegazione comparata con codice eseguibile.

In [ ]:
# Esempio 1: Java e parallelismo
risposta_java = ask("Devo ordinare 10 milioni di interi in Java su una macchina multi-core")

In [ ]:
# Esempio 2: Top-K in Python su dataset enorme
risposta_python = ask("Mi servono solo i 10 elementi più grandi di una lista enorme in Python senza ordinare tutto")

In [ ]:
# Esempio 3: Ordinamento in C++ con garanzia di stabilità
risposta_cpp = ask("In C++ ho oggetti complessi e devo mantenere l'ordine relativo degli elementi con chiave identica")